# Transformer Model Inference on Kaggle

This notebook performs inference using a pre-trained transformer model for Finnish-English translation.

**Required inputs:**
- `.pt` model file (trained transformer model)
- Dataset split folder (as created by train.py)
- Source code files (utils.py, encoder.py, decoder.py, train.py, config.py)

**Features:**
- Three decoding strategies: Greedy, Beam Search, Top-k Sampling
- BLEU score evaluation
- Custom text translation
- Downloadable results

## 1. Install Required Dependencies

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install tokenizers
!pip install nltk
!pip install torchmetrics
!pip install tqdm

# Download NLTK data
import nltk
nltk.download('punkt')
nltk.download('bleu')

print("✅ All dependencies installed successfully!")

## 2. Import Libraries and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import numpy as np
import pandas as pd
from pathlib import Path
import json
import pickle
import zipfile
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# NLTK imports
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("\n✅ Libraries imported and setup complete!")

## 3. Upload and Extract Model Files

**Instructions:**
1. Upload your `.pt` model file to Kaggle
2. Upload your dataset splits folder (zip file)
3. Upload your source code files (utils.py, encoder.py, decoder.py, train.py, config.py)
4. Update the file paths below

In [ ]:
# Configure file paths - UPDATE THESE PATHS
MODEL_PATH = "/kaggle/input/your-model/tmodel_14.pt"  # Update this path
DATASET_SPLITS_ZIP = "/kaggle/input/your-dataset/dataset_splits.zip"  # Update this path
SOURCE_CODE_DIR = "/kaggle/input/your-source-code/"  # Update this path

# Create working directory
work_dir = Path("/kaggle/working")
work_dir.mkdir(exist_ok=True)

# Extract dataset splits
if Path(DATASET_SPLITS_ZIP).exists():
    print("Extracting dataset splits...")
    with zipfile.ZipFile(DATASET_SPLITS_ZIP, 'r') as zip_ref:
        zip_ref.extractall(work_dir)
    print("✅ Dataset splits extracted")
else:
    print("❌ Dataset splits file not found. Please upload and update the path.")

# Copy source code files
import shutil
source_files = ['utils.py', 'encoder.py', 'decoder.py', 'train.py', 'config.py']

for file in source_files:
    src_path = Path(SOURCE_CODE_DIR) / file
    dst_path = work_dir / file
    if src_path.exists():
        shutil.copy2(src_path, dst_path)
        print(f"✅ Copied {file}")
    else:
        print(f"❌ {file} not found in {SOURCE_CODE_DIR}")

# Change to working directory
os.chdir(work_dir)
print(f"\nWorking directory: {os.getcwd()}")
print("Files in working directory:")
for item in os.listdir('.'):
    print(f"  {item}")

## 4. Load Dataset Splits

In [ ]:
# Import custom modules
from utils import (
    BilingualDataset, load_custom_dataset, get_or_build_tokenizer, greedy_decode, causal_mask,
    InputEmbedding, PositionalEncoding, RotaryPositionalEmbedding, RelativePositionalBias, ProjectionLayer,
    MultiHeadAttention, feedForwardBlock
)
from encoder import Encoder, EncoderBlock
from decoder import Decoder, DecoderBlock
from train import Transformer, build_transformer
from config import get_config, get_weights_file_path

def load_dataset_from_splits(split_dir="dataset_splits", split_type="test"):
    """
    Load dataset from saved splits
    """
    split_path = Path(split_dir)
    
    if not split_path.exists():
        raise FileNotFoundError(f"Split directory not found: {split_path}")
    
    # Try to load from pickle file first
    pickle_file = split_path / "partitions_complete_latest.pkl"
    if pickle_file.exists():
        print(f"Loading dataset splits from pickle: {pickle_file}")
        try:
            with open(pickle_file, 'rb') as f:
                data = pickle.load(f)
            
            if split_type == "test":
                return data['test_ds_raw']
            elif split_type == "validation":
                return data['val_ds_raw']
            elif split_type == "train":
                return data['train_ds_raw']
            else:
                raise ValueError(f"Invalid split_type: {split_type}")
                
        except Exception as e:
            print(f"Warning: Could not load from pickle file: {e}")
            print("Falling back to loading from original dataset with indices...")
    
    # Fallback: load indices and reconstruct from original dataset
    indices_file = split_path / "partition_indices_latest.json"
    if not indices_file.exists():
        raise FileNotFoundError(f"No partition indices found: {indices_file}")
    
    print(f"Loading dataset indices from: {indices_file}")
    with open(indices_file, 'r') as f:
        indices_data = json.load(f)
    
    # Load original dataset (should be uploaded to Kaggle)
    print("Loading original dataset...")
    # These files should be uploaded as part of your dataset
    fi_file_path = "datasets/EUbookshop.fi"
    en_file_path = "datasets/EUbookshop.en" 
    
    if not Path(fi_file_path).exists() or not Path(en_file_path).exists():
        raise FileNotFoundError("Original dataset files not found. Please upload datasets/EUbookshop.fi and datasets/EUbookshop.en")
    
    ds_raw = load_custom_dataset(fi_file_path, en_file_path)
    
    # Get the appropriate indices
    if split_type == "test":load
        split_indices = indices_data['test_indices']
    elif split_type == "validation":
        split_indices = indices_data['val_indices']
    elif split_type == "train":
        split_indices = indices_data['train_indices']
    else:
        raise ValueError(f"Invalid split_type: {split_type}")
    
    # Create subset using indices
    from torch.utils.data import Subset
    split_dataset = Subset(ds_raw, split_indices)
    print(f"Loaded {split_type} split with {len(split_dataset)} samples")
    
    return split_dataset

# Load configuration
config = get_config()
print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load dataset splits
try:
    test_dataset_raw = load_dataset_from_splits("dataset_splits", "test")
    print(f"\n✅ Successfully loaded test dataset")
    print(f"   Test dataset size: {len(test_dataset_raw)}")
except Exception as e:
    print(f"❌ Error loading dataset splits: {e}")
    test_dataset_raw = None

## 5. Initialize Tokenizers

In [ ]:
# Load tokenizers
print("Loading tokenizers...")

# Check if tokenizer files exist
tokenizer_src_file = f"tokenizer_{config['lang_src']}.json"
tokenizer_tgt_file = f"tokenizer_{config['lang_tgt']}.json"

if Path(tokenizer_src_file).exists() and Path(tokenizer_tgt_file).exists():
    print(f"Found existing tokenizer files:")
    print(f"  Source: {tokenizer_src_file}")
    print(f"  Target: {tokenizer_tgt_file}")
    
    # Load tokenizers directly
    from tokenizers import Tokenizer
    tokenizer_src = Tokenizer.from_file(tokenizer_src_file)
    tokenizer_tgt = Tokenizer.from_file(tokenizer_tgt_file)
    
    print(f"✅ Tokenizers loaded successfully")
    print(f"   Source vocabulary size: {tokenizer_src.get_vocab_size()}")
    print(f"   Target vocabulary size: {tokenizer_tgt.get_vocab_size()}")
    
else:
    print("❌ Tokenizer files not found. Building tokenizers from dataset...")
    
    if test_dataset_raw is not None:
        # Need to load full dataset for tokenizer building
        fi_file_path = "datasets/EUbookshop.fi"
        en_file_path = "datasets/EUbookshop.en"
        
        if Path(fi_file_path).exists() and Path(en_file_path).exists():
            print("Loading full dataset for tokenizer building...")
            ds_raw_full = load_custom_dataset(fi_file_path, en_file_path)
            
            tokenizer_src = get_or_build_tokenizer(config, ds_raw_full, config['lang_src'])
            tokenizer_tgt = get_or_build_tokenizer(config, ds_raw_full, config['lang_tgt'])
            
            print(f"✅ Tokenizers built successfully")
            print(f"   Source vocabulary size: {tokenizer_src.get_vocab_size()}")
            print(f"   Target vocabulary size: {tokenizer_tgt.get_vocab_size()}")
        else:
            print("❌ Original dataset files not found for tokenizer building")
            tokenizer_src = None
            tokenizer_tgt = None
    else:
        print("❌ No dataset available for tokenizer building")
        tokenizer_src = None
        tokenizer_tgt = None

# Create BilingualDataset for evaluation if we have everything
test_dataset = None
test_dataloader = None

if test_dataset_raw is not None and tokenizer_src is not None and tokenizer_tgt is not None:
    print("\nCreating BilingualDataset for evaluation...")
    
    test_dataset = BilingualDataset(
        test_dataset_raw, 
        tokenizer_src, 
        tokenizer_tgt, 
        config['lang_src'], 
        config['lang_tgt'], 
        config['seq_len']
    )
    
    # Create test dataloader (batch_size=1 for inference)
    test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    print(f"✅ BilingualDataset created successfully")
    print(f"   Test samples: {len(test_dataset)}")
    
else:
    print("❌ Cannot create BilingualDataset - missing components")

# Test tokenizers
if tokenizer_src and tokenizer_tgt:
    print("\nTesting tokenizers:")
    test_fi = "Hei, kuinka voit?"
    test_en = "Hello, how are you?"
    
    fi_tokens = tokenizer_src.encode(test_fi)
    en_tokens = tokenizer_tgt.encode(test_en)
    
    print(f"Finnish: '{test_fi}' -> {fi_tokens.tokens}")
    print(f"English: '{test_en}' -> {en_tokens.tokens}")

## 6. Load Trained Model

In [ ]:
def load_trained_model(config, model_path, vocab_src_len, vocab_tgt_len, device):
    """Load a trained transformer model from checkpoint"""
    print(f"Loading model from {model_path}")
    
    if not Path(model_path).exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")
    
    # Get positional encoding configuration
    positional_encoding = config.get('positional_encoding', None)
    relative_bias_clamp_distance = config.get('relative_bias_clamp_distance', 32)
    
    # Legacy support for use_rope parameter
    use_rope = config.get('use_rope', False) if positional_encoding is None else None
    
    # Build the model with same architecture as training
    model = build_transformer(
        vocab_src_len, vocab_tgt_len, 
        config['seq_len'], config['seq_len'], 
        config['d_model'],
        N=2,  # Number of encoder/decoder layers
        d_ff=1024,  # Feed forward dimension
        h=2,  # Number of attention heads
        use_rope=use_rope,
        positional_encoding=positional_encoding,
        relative_bias_clamp_distance=relative_bias_clamp_distance
    )
    
    # Load the saved state
    print("Loading model weights...")
    state = torch.load(model_path, map_location=device)
    
    # Handle both DataParallel and regular model states
    if 'model_state_dict' in state:
        model_state_dict = state['model_state_dict']
        
        # Remove 'module.' prefix if present (from DataParallel training)
        if list(model_state_dict.keys())[0].startswith('module.'):
            model_state_dict = {k.replace('module.', ''): v for k, v in model_state_dict.items()}
        
        model.load_state_dict(model_state_dict)
        print(f"✅ Loaded model from epoch {state.get('epoch', 'unknown')}")
        
        # Print additional info if available
        if 'loss' in state:
            print(f"   Training loss: {state['loss']:.4f}")
        if 'global_step' in state:
            print(f"   Global step: {state['global_step']}")
            
    else:
        # Direct state dict
        model.load_state_dict(state)
        print("✅ Loaded model state dict directly")
    
    model = model.to(device)
    model.eval()
    
    # Print model info
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Total parameters: {total_params:,}")
    print(f"   Trainable parameters: {trainable_params:,}")
    
    return model

# Load the model
if tokenizer_src and tokenizer_tgt:
    try:
        model = load_trained_model(
            config, 
            MODEL_PATH, 
            tokenizer_src.get_vocab_size(), 
            tokenizer_tgt.get_vocab_size(), 
            device
        )
        print("\n✅ Model loaded successfully!")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        model = None
else:
    print("❌ Cannot load model without tokenizers")
    model = None

## 7. Implement Decoding Strategies

## 7.5. GPU Optimization & Performance Settings

**⚡ Critical for Kaggle GPU performance!**

In [ ]:
# ⚡ CRITICAL GPU OPTIMIZATION FOR KAGGLE
print("🔥 Setting up GPU optimizations for maximum speed...")

if torch.cuda.is_available() and model is not None:
    # ✅ Ensure model is on GPU
    model = model.to(device)
    print(f"✅ Model moved to: {device}")
    
    # ✅ Enable optimizations
    torch.backends.cudnn.benchmark = True  # Optimize for consistent input sizes
    torch.backends.cudnn.deterministic = False  # Allow non-deterministic for speed
    
    # ✅ Set optimal GPU settings
    if hasattr(torch.cuda, 'set_per_process_memory_fraction'):
        torch.cuda.set_per_process_memory_fraction(0.9)  # Use 90% of GPU memory
    
    # ✅ Clear any existing cache
    torch.cuda.empty_cache()
    
    # ✅ Verify setup
    print("🚀 GPU Performance Optimizations:")
    print(f"   Device: {torch.cuda.get_device_name()}")
    print(f"   Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"   Model Device: {next(model.parameters()).device}")
    print(f"   CUDNN Benchmark: {torch.backends.cudnn.benchmark}")
    print(f"   Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    
    # ✅ Quick inference test to warm up GPU
    print("\n🔥 Warming up GPU with test inference...")
    if tokenizer_src and tokenizer_tgt:
        try:
            test_input = torch.randint(0, 100, (1, 50)).to(device)
            test_mask = torch.ones(1, 1, 1, 50).to(device)
            with torch.no_grad():
                _ = model.encode(test_input, test_mask)
            print("✅ GPU warmup successful!")
            torch.cuda.empty_cache()
        except Exception as e:
            print(f"⚠️ GPU warmup failed: {e}")
    
    print(f"\n🎯 GPU Setup Complete! Ready for FAST inference.")
    print("⚡ Expected speed: ~100-500 samples/minute (depending on sequence length)")
    
else:
    print("❌ GPU not available or model not loaded!")
    print("⚠️ WARNING: CPU inference will take 9+ hours!")
    print("🔧 Make sure you're using Kaggle GPU accelerator")

# ✅ Set inference parameters for optimal GPU performance
OPTIMAL_BATCH_SIZE = 1  # Keep at 1 for beam search compatibility
GPU_MEMORY_CLEANUP_INTERVAL = 50  # Clean GPU memory every N samples

print(f"\n⚙️ Optimal settings configured:")
print(f"   Batch Size: {OPTIMAL_BATCH_SIZE}")
print(f"   GPU Cleanup Interval: {GPU_MEMORY_CLEANUP_INTERVAL}")
print(f"   Ready for evaluation!")

In [ ]:
def beam_search_decode(model, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device, beam_size=2):
    """GPU-optimized Beam search decoding"""
    batch_size = source.size(0)
    if batch_size != 1:
        raise ValueError(f"beam_search_decode only supports batch_size=1, got {batch_size}")
    
    sos_idx = tokenizer_tgt.token_to_id('[SOS]')
    eos_idx = tokenizer_tgt.token_to_id('[EOS]')
    
    # ✅ Encode the source sequence - KEEP ON GPU
    encoder_output = model.encode(source, source_mask)
    
    # ✅ Initialize beams with SOS token - ON GPU
    initial_sequence = torch.tensor([sos_idx], dtype=torch.long, device=device)
    beams = [(initial_sequence, 0.0)]
    completed_beams = []
    
    for step in range(max_len):
        new_beams = []
        
        for sequence, log_prob in beams:
            if sequence[-1] == eos_idx:
                completed_beams.append((sequence, log_prob))
                continue
            
            # ✅ Keep decoder operations on GPU
            decoder_input = sequence.unsqueeze(0)  # Already on GPU
            decoder_mask = causal_mask(decoder_input.size(1)).type_as(source_mask).to(device)
            
            # ✅ All model operations stay on GPU
            out = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)
            logits = model.project(out[:, -1])
            log_probs = F.log_softmax(logits, dim=-1).squeeze(0)
            
            # ✅ Top-k selection on GPU
            top_log_probs, top_indices = torch.topk(log_probs, beam_size)
            
            for i in range(beam_size):
                new_token = top_indices[i].item()
                new_log_prob = log_prob + top_log_probs[i].item()
                # ✅ Keep sequence construction on GPU
                new_sequence = torch.cat([sequence, torch.tensor([new_token], dtype=torch.long, device=device)])
                new_beams.append((new_sequence, new_log_prob))
        
        if new_beams:
            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]
        
        if not beams:
            break
    
    completed_beams.extend(beams)
    
    if completed_beams:
        best_sequence, _ = max(completed_beams, key=lambda x: x[1])
        return best_sequence  # Return GPU tensor
    else:
        return greedy_decode(model, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device)


def top_k_sampling_decode(model, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device, k=50, temperature=1.0):
    """GPU-optimized Top-k sampling decoding"""
    batch_size = source.size(0)
    if batch_size != 1:
        raise ValueError(f"top_k_sampling_decode only supports batch_size=1, got {batch_size}")
    
    sos_idx = tokenizer_tgt.token_to_id('[SOS]')
    eos_idx = tokenizer_tgt.token_to_id('[EOS]')
    
    # ✅ Encode source and initialize decoder input on GPU
    encoder_output = model.encode(source, source_mask)
    decoder_input = torch.empty(1, 1, dtype=torch.long, device=device).fill_(sos_idx)
    
    for step in range(max_len):
        if decoder_input.size(1) == max_len:
            break
            
        # ✅ All operations stay on GPU
        decoder_mask = causal_mask(decoder_input.size(1)).type_as(source_mask).to(device)
        out = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)
        logits = model.project(out[:, -1]) / temperature
        
        # ✅ Top-k sampling on GPU
        top_logits, top_indices = torch.topk(logits, k, dim=-1)
        top_probs = F.softmax(top_logits, dim=-1)
        sampled_index = torch.multinomial(top_probs, 1)
        next_word = top_indices.gather(-1, sampled_index).squeeze()
        
        # ✅ Extend sequence on GPU
        next_word_tensor = torch.empty(1, 1, dtype=torch.long, device=device).fill_(next_word.item())
        decoder_input = torch.cat([decoder_input, next_word_tensor], dim=1)
        
        if next_word == eos_idx:
            break
    
    return decoder_input.squeeze(0)  # Return GPU tensor


# ✅ Add GPU verification function
def verify_gpu_setup():
    """Verify GPU setup and model placement"""
    if torch.cuda.is_available():
        print("🔥 GPU Setup Verification:")
        print(f"   Device: {torch.cuda.get_device_name()}")
        print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
        print(f"   Available Memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
        
        if 'model' in globals() and model is not None:
            model_device = next(model.parameters()).device
            print(f"   Model Device: {model_device}")
            print(f"   Model on GPU: {'✅' if model_device.type == 'cuda' else '❌'}")
        
        torch.cuda.empty_cache()
        print("   GPU Cache Cleared ✅")
        return True
    else:
        print("❌ GPU not available!")
        return False

# Run GPU verification
gpu_available = verify_gpu_setup()

print("✅ GPU-optimized decoding strategies implemented!")
print("🚀 Key optimizations:")
print("   - All tensor operations stay on GPU")
print("   - Minimal CPU transfers")
print("   - GPU memory management")
print("   - Optimized beam search and top-k sampling")

if gpu_available:
    print("\n🎯 Ready for high-speed GPU inference!")
else:
    print("\n⚠️ Will run on CPU (much slower)")

## 8. Run Model Inference and Evaluation

**Note:** This section creates a sample test to demonstrate functionality. For full evaluation, you would need your complete test dataset.

In [ ]:
def translate_text(model, text, tokenizer_src, tokenizer_tgt, config, device, method='greedy'):
    """Translate a single text using specified method"""
    model.eval()
    
    with torch.no_grad():
        # Tokenize input
        input_tokens = tokenizer_src.encode(text).ids
        
        # Check length
        if len(input_tokens) + 2 > config['seq_len']:
            return f"Text too long ({len(input_tokens)} tokens)"
        
        # Create encoder input
        sos_token = torch.tensor([tokenizer_src.token_to_id('[SOS]')], dtype=torch.int64)
        eos_token = torch.tensor([tokenizer_src.token_to_id('[EOS]')], dtype=torch.int64)
        pad_token = torch.tensor([tokenizer_src.token_to_id('[PAD]')], dtype=torch.int64)
        
        padding_length = config['seq_len'] - len(input_tokens) - 2
        
        encoder_input = torch.cat([
            sos_token,
            torch.tensor(input_tokens, dtype=torch.int64),
            eos_token,
            torch.tensor([pad_token] * padding_length, dtype=torch.int64)
        ]).unsqueeze(0).to(device)
        
        # Create encoder mask
        encoder_mask = (encoder_input != pad_token).unsqueeze(0).unsqueeze(0).int().to(device)
        
        # Generate translation based on method
        if method == 'greedy':
            model_out = greedy_decode(model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, config['seq_len'], device)
        elif method == 'beam_search':
            model_out = beam_search_decode(model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, config['seq_len'], device, beam_size=5)
        elif method == 'top_k':
            model_out = top_k_sampling_decode(model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, config['seq_len'], device, k=50, temperature=0.8)
        else:
            raise ValueError(f"Unknown method: {method}")
        
        # Decode output
        translation = tokenizer_tgt.decode(model_out.detach().cpu().numpy())
        
        return translation

def evaluate_single_strategy_fast(model, test_dataloader, tokenizer_src, tokenizer_tgt, config, device, strategy='greedy', max_samples=None):
    """
    🚀 FAST evaluation with SINGLE strategy - much faster than running all three!
    """
    model.eval()
    
    print(f"⚡ FAST SINGLE-STRATEGY EVALUATION: {strategy.upper()}")
    print(f"Device: {device}")
    print("=" * 60)
    
    # Choose decoding function
    if strategy == 'greedy':
        decode_func = greedy_decode
        print("🏃‍♂️ Using GREEDY decoding (fastest)")
    elif strategy == 'beam_search':
        decode_func = lambda model, enc_input, enc_mask, tok_src, tok_tgt, max_len, dev: beam_search_decode(model, enc_input, enc_mask, tok_src, tok_tgt, max_len, dev, beam_size=2)
        print("🔍 Using BEAM SEARCH decoding (slower but better quality)")
    elif strategy == 'top_k':
        decode_func = lambda model, enc_input, enc_mask, tok_src, tok_tgt, max_len, dev: top_k_sampling_decode(model, enc_input, enc_mask, tok_src, tok_tgt, max_len, dev, k=5, temperature=0.8)
        print("🎲 Using TOP-K SAMPLING decoding (diverse outputs)")
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
    
    # Storage for results
    predictions = []
    references = []
    sources = []
    
    dataset_size = len(test_dataloader)
    if max_samples is not None:
        dataset_size = min(max_samples, dataset_size)
    
    print(f"⚡ Processing {dataset_size} samples with {strategy.upper()}...")
    
    # Pre-cache token IDs
    sos_id = tokenizer_tgt.token_to_id('[SOS]')
    eos_id = tokenizer_tgt.token_to_id('[EOS]')
    pad_id = tokenizer_tgt.token_to_id('[PAD]')
    
    successful_samples = 0
    skipped_samples = 0
    
    import time
    start_time = time.time()
    
    with torch.no_grad():
        for i, batch in enumerate(tqdm(test_dataloader, desc=f"{strategy.upper()} Inference", total=dataset_size)):
            if max_samples is not None and i >= max_samples:
                break
                
            try:
                # ✅ GPU tensors
                encoder_input = batch["encoder_input"].to(device, non_blocking=True)
                encoder_mask = batch["encoder_mask"].to(device, non_blocking=True)
                label = batch['label'].to(device, non_blocking=True)
                
                if encoder_input.size(0) != 1:
                    skipped_samples += 1
                    continue
                
                # ✅ Extract reference tokens efficiently
                reference_tokens = []
                label_cpu = label[0].cpu()
                for token_id in label_cpu:
                    token_id_val = token_id.item()
                    if token_id_val == eos_id:
                        break
                    if token_id_val not in [sos_id, pad_id]:
                        reference_tokens.append(tokenizer_tgt.id_to_token(token_id_val))
                
                if not reference_tokens:
                    skipped_samples += 1
                    continue
                
                source_text = batch["src_text"][0]
                
                # ✅ Single strategy inference - MUCH FASTER!
                output = decode_func(model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, config['seq_len'], device)
                
                # ✅ Extract prediction tokens
                pred_tokens = []
                output_cpu = output.cpu() if output.is_cuda else output
                for token_id in output_cpu:
                    token_id_val = token_id.item()
                    if token_id_val == eos_id:
                        break
                    if token_id_val not in [sos_id, pad_id]:
                        pred_tokens.append(tokenizer_tgt.id_to_token(token_id_val))
                
                if pred_tokens:
                    predictions.append(pred_tokens)
                    references.append([reference_tokens])  # List of lists for BLEU
                    sources.append(source_text)
                    successful_samples += 1
                
                # ✅ Progress updates with speed
                if (i + 1) % 500 == 0:
                    elapsed = time.time() - start_time
                    samples_per_sec = (i + 1) / elapsed
                    remaining_samples = dataset_size - (i + 1)
                    eta_seconds = remaining_samples / samples_per_sec
                    eta_minutes = eta_seconds / 60
                    
                    if torch.cuda.is_available():
                        gpu_memory = torch.cuda.memory_allocated() / 1024**3
                        print(f"\n🔥 Progress Update - Sample {i+1}/{dataset_size}")
                        print(f"   Speed: {samples_per_sec:.1f} samples/sec")
                        print(f"   ETA: {eta_minutes:.1f} minutes")
                        print(f"   GPU Memory: {gpu_memory:.2f} GB")
                        print(f"   Success: {successful_samples}, Skipped: {skipped_samples}")
                
                # ✅ Memory cleanup
                if i % 100 == 0 and torch.cuda.is_available():
                    torch.cuda.empty_cache()
                
            except Exception as e:
                print(f"Error processing sample {i}: {e}")
                skipped_samples += 1
                continue
    
    # ✅ Calculate BLEU score
    bleu_score = 0.0
    if predictions and references:
        from nltk.translate.bleu_score import corpus_bleu
        bleu_score = corpus_bleu(references, predictions)
    
    # ✅ Final stats
    total_time = time.time() - start_time
    avg_time_per_sample = total_time / max(successful_samples, 1)
    
    print(f"\n🎯 {strategy.upper()} EVALUATION COMPLETE!")
    print(f"   Total Time: {total_time/60:.2f} minutes")
    print(f"   Successful Samples: {successful_samples}")
    print(f"   Average Time/Sample: {avg_time_per_sample:.3f} seconds")
    print(f"   Speed: {successful_samples/total_time:.1f} samples/second")
    print(f"   BLEU Score: {bleu_score:.4f}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return {
        'strategy': strategy,
        'predictions': predictions,
        'references': references,
        'sources': sources,
        'bleu_score': bleu_score,
        'successful_samples': successful_samples,
        'total_time_minutes': total_time/60,
        'samples_per_second': successful_samples/total_time
    }

# 🚀 CHOOSE YOUR EVALUATION STRATEGY - MUCH FASTER!
if model and test_dataloader and tokenizer_src and tokenizer_tgt:
    print("🚀 FAST SINGLE-STRATEGY EVALUATION OPTIONS:")
    print("=" * 50)
    print("Choose ONE strategy for fast evaluation:")
    print("1. 'greedy' - Fastest (~5-10 minutes for full dataset)")
    print("2. 'beam_search' - Medium speed, better quality (~15-30 minutes)")  
    print("3. 'top_k' - Medium speed, diverse outputs (~15-30 minutes)")
    print("\n⚡ Running GREEDY (fastest) by default...")
    
    # ✅ Verify GPU setup
    if torch.cuda.is_available():
        print(f"\n✅ GPU Device: {torch.cuda.get_device_name()}")
        print(f"✅ Model on GPU: {next(model.parameters()).device}")
        torch.cuda.empty_cache()
    
    # 🚀 FAST SINGLE STRATEGY EVALUATION
    # Change 'greedy' to 'beam_search' or 'top_k' if you want different strategy
    evaluation_results = evaluate_single_strategy_fast(
        model, test_dataloader, tokenizer_src, tokenizer_tgt, config, device,
        strategy='top_k',  # 🔥 CHANGE THIS: 'greedy', 'beam_search', or 'top_k'
        max_samples=1000    # Set to 1000 for quick test, None for full dataset
    )
    
    print(f"\n✅ Single strategy evaluation completed!")
    print(f"🏆 {evaluation_results['strategy'].upper()} BLEU: {evaluation_results['bleu_score']:.4f}")
    print(f"⚡ Speed: {evaluation_results['samples_per_second']:.1f} samples/second")
    
else:
    print("❌ Cannot run evaluation - missing components")
    evaluation_results = None

# 🔥 OPTIONAL: Run all three strategies (will take 3x longer)
run_all_strategies = False  # Set to True if you want to compare all strategies

if run_all_strategies and evaluation_results:
    print("\n🔄 RUNNING ALL THREE STRATEGIES FOR COMPARISON...")
    print("⚠️ This will take 3x longer!")
    
    all_results = {}
    for strategy in ['greedy', 'beam_search', 'top_k']:
        print(f"\n🚀 Running {strategy.upper()}...")
        result = evaluate_single_strategy_fast(
            model, test_dataloader, tokenizer_src, tokenizer_tgt, config, device,
            strategy=strategy, max_samples=None
        )
        all_results[strategy] = result
    
    print("\n🏆 FINAL COMPARISON:")
    print("=" * 40)
    for strategy, result in all_results.items():
        print(f"{strategy.upper():12}: BLEU {result['bleu_score']:.4f} | Speed {result['samples_per_second']:.1f} samp/sec")
    
    # 📊 SAVE TABULATED BLEU SCORES TO FILE
    print("\n💾 Saving tabulated BLEU scores...")
    
    # Create results directory if it doesn't exist
    results_dir = Path("inference_results")
    results_dir.mkdir(exist_ok=True)
    
    # Save as nicely formatted table
    bleu_table_file = results_dir / "bleu_scores_comparison_table.txt"
    with open(bleu_table_file, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("TRANSFORMER MODEL - BLEU SCORE COMPARISON TABLE\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"Model: {MODEL_PATH}\n")
        f.write(f"Language Pair: {config['lang_src']} → {config['lang_tgt']}\n")
        f.write(f"Test Dataset Size: {len(test_dataloader)} samples\n")
        f.write(f"Evaluation Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("RESULTS SUMMARY:\n")
        f.write("-" * 80 + "\n")
        f.write(f"{'Strategy':<15} {'BLEU Score':<12} {'Samples':<8} {'Time (min)':<12} {'Speed (samp/sec)':<15}\n")
        f.write("-" * 80 + "\n")
        
        for strategy, result in all_results.items():
            f.write(f"{strategy.upper():<15} "
                   f"{result['bleu_score']:<12.4f} "
                   f"{result['successful_samples']:<8} "
                   f"{result['total_time_minutes']:<12.2f} "
                   f"{result['samples_per_second']:<15.1f}\n")
        
        f.write("-" * 80 + "\n")
        
        # Find best strategy
        best_strategy = max(all_results.items(), key=lambda x: x[1]['bleu_score'])
        f.write(f"\n🏆 BEST PERFORMING STRATEGY: {best_strategy[0].upper()}\n")
        f.write(f"   BLEU Score: {best_strategy[1]['bleu_score']:.4f}\n")
        f.write(f"   Evaluation Time: {best_strategy[1]['total_time_minutes']:.2f} minutes\n")
        f.write(f"   Speed: {best_strategy[1]['samples_per_second']:.1f} samples/second\n\n")
        
        # Additional statistics
        all_bleu_scores = [result['bleu_score'] for result in all_results.values()]
        f.write("STATISTICS:\n")
        f.write("-" * 30 + "\n")
        f.write(f"Highest BLEU: {max(all_bleu_scores):.4f}\n")
        f.write(f"Lowest BLEU:  {min(all_bleu_scores):.4f}\n")
        f.write(f"Average BLEU: {sum(all_bleu_scores)/len(all_bleu_scores):.4f}\n")
        f.write(f"BLEU Range:   {max(all_bleu_scores) - min(all_bleu_scores):.4f}\n")
        
        f.write("\n" + "=" * 80 + "\n")
    
    # Also save as CSV for easy import into Excel/analysis tools
    bleu_csv_file = results_dir / "bleu_scores_comparison.csv"
    comparison_data = []
    for strategy, result in all_results.items():
        comparison_data.append({
            'Strategy': strategy.upper(),
            'BLEU_Score': result['bleu_score'],
            'Successful_Samples': result['successful_samples'],
            'Total_Time_Minutes': result['total_time_minutes'],
            'Samples_Per_Second': result['samples_per_second']
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df.to_csv(bleu_csv_file, index=False)
    
    print(f"✅ BLEU comparison table saved to: {bleu_table_file}")
    print(f"✅ BLEU comparison CSV saved to: {bleu_csv_file}")
    print(f"\n📊 Quick Summary:")
    print(comparison_df.to_string(index=False, float_format='%.4f'))
    
    # Update evaluation_results with all strategies
    evaluation_results = all_results

## 9. Evaluate Model Performance

This section would normally evaluate on your full test dataset. For demonstration, we'll show the structure.

In [ ]:
def calculate_bleu_scores(predictions, references):
    """Calculate BLEU scores for predictions"""
    if not predictions or not references:
        return 0.0
    
    # Calculate BLEU score using corpus_bleu
    bleu_score = corpus_bleu(references, predictions)
    return bleu_score

# Calculate BLEU scores for all three strategies
if evaluation_results:
    print("Calculating BLEU Scores for Full Dataset Evaluation:")
    print("=" * 60)
    
    bleu_scores = {}
    
    for strategy in ['greedy', 'beam_search', 'top_k_sampling']:
        predictions = evaluation_results[strategy]['predictions']
        references = evaluation_results[strategy]['references']
        sources = evaluation_results[strategy]['sources']
        
        if predictions and references:
            # Calculate corpus BLEU
            bleu_score = calculate_bleu_scores(predictions, references)
            bleu_scores[strategy] = bleu_score
            
            print(f"\n{strategy.upper()} Results:")
            print(f"  Samples evaluated: {len(predictions)}")
            print(f"  BLEU Score: {bleu_score:.4f}")
            
            # Show a few examples
            print("  Sample translations:")
            for i in range(min(3, len(predictions))):
                print(f"    Source: {sources[i]}")
                print(f"    Reference: {' '.join(references[i][0])}")
                print(f"    Prediction: {' '.join(predictions[i])}")
                print()
        else:
            bleu_scores[strategy] = 0.0
            print(f"\n{strategy.upper()}: No valid predictions generated")
    
    # Create detailed comparison DataFrame
    if all(len(evaluation_results[strategy]['predictions']) > 0 for strategy in ['greedy', 'beam_search', 'top_k_sampling']):
        # Take minimum length to ensure all strategies have same number of samples
        min_samples = min(len(evaluation_results[strategy]['predictions']) for strategy in ['greedy', 'beam_search', 'top_k_sampling'])
        
        comparison_data = {
            'Source': evaluation_results['greedy']['sources'][:min_samples],
            'Reference': [' '.join(ref[0]) for ref in evaluation_results['greedy']['references'][:min_samples]],
            'Greedy': [' '.join(pred) for pred in evaluation_results['greedy']['predictions'][:min_samples]],
            'Beam_Search': [' '.join(pred) for pred in evaluation_results['beam_search']['predictions'][:min_samples]],
            'Top_k': [' '.join(pred) for pred in evaluation_results['top_k_sampling']['predictions'][:min_samples]]
        }
        
        comparison_df = pd.DataFrame(comparison_data)
        
        print(f"\nFull Dataset Evaluation Summary:")
        print("=" * 40)
        for strategy, score in bleu_scores.items():
            print(f"{strategy.upper():15}: {score:.4f}")
        
        print(f"\nFirst 10 sample comparisons:")
        print(comparison_df.head(10).to_string(index=False, max_colwidth=50))
        
    else:
        print("Cannot create comparison table - unequal number of predictions across strategies")
        comparison_df = None
        
else:
    print("No evaluation results to analyze")
    bleu_scores = {}
    comparison_df = None

print("\n✅ BLEU score calculation completed!")

## 10. Test Custom Examples

Enter your own Finnish text to see translations:

In [ ]:
# Additional Custom Examples (Supplementary to main evaluation)
def test_custom_examples():
    """Test model on additional custom examples"""
    if not (model and tokenizer_src and tokenizer_tgt):
        print("❌ Model or tokenizers not loaded")
        return []
    
    print("Testing Additional Custom Examples:")
    print("=" * 40)
    
    # Additional test examples beyond the main dataset
    custom_examples = [
        "Minä rakastan sinua.",
        "Mitä kuuluu?", 
        "Hyvää huomenta!",
        "Nähdään myöhemmin.",
        "Anteeksi, en ymmärrä.",
        "Kuinka paljon tämä maksaa?",
        "Voinko auttaa sinua?"
    ]
    
    custom_results = []
    
    for i, text in enumerate(custom_examples):
        print(f"\nCustom Example {i+1}: {text}")
        
        try:
            # Test all three methods
            greedy_result = translate_text(model, text, tokenizer_src, tokenizer_tgt, config, device, 'greedy')
            beam_result = translate_text(model, text, tokenizer_src, tokenizer_tgt, config, device, 'beam_search')
            topk_result = translate_text(model, text, tokenizer_src, tokenizer_tgt, config, device, 'top_k')
            
            print(f"  Greedy:      {greedy_result}")
            print(f"  Beam Search: {beam_result}")
            print(f"  Top-k:       {topk_result}")
            
            custom_results.append({
                'finnish': text,
                'greedy': greedy_result,
                'beam_search': beam_result,
                'top_k': topk_result
            })
            
        except Exception as e:
            print(f"  Error: {e}")
    
    return custom_results

# Run custom examples if main evaluation was successful
if evaluation_results:
    print("\nRunning additional custom examples...")
    custom_results = test_custom_examples()
    print(f"\n✅ Custom examples completed! ({len(custom_results)} examples)")
else:
    print("Skipping custom examples - main evaluation not completed")
    custom_results = []

## 11. Save Results

In [ ]:
# Save complete evaluation results to files
if evaluation_results:
    # Create results directory
    results_dir = Path("inference_results")
    results_dir.mkdir(exist_ok=True)
    
    print("Saving complete evaluation results...")
    
    # Save detailed evaluation results as JSON
    detailed_results_file = results_dir / "detailed_evaluation_results.json"
    
    # Convert evaluation results to serializable format
    serializable_results = {}
    for strategy in ['greedy', 'beam_search', 'top_k_sampling']:
        serializable_results[strategy] = {
            'predictions': [' '.join(pred) for pred in evaluation_results[strategy]['predictions']],
            'references': [' '.join(ref[0]) for ref in evaluation_results[strategy]['references']],
            'sources': evaluation_results[strategy]['sources'],
            'bleu_score': bleu_scores.get(strategy, 0.0),
            'sample_count': len(evaluation_results[strategy]['predictions'])
        }
    
    with open(detailed_results_file, 'w', encoding='utf-8') as f:
        json.dump(serializable_results, f, ensure_ascii=False, indent=2)
    print(f"✅ Detailed evaluation results saved to: {detailed_results_file}")
    
    # Save BLEU scores summary
    bleu_summary_file = results_dir / "bleu_scores_summary.json"
    bleu_summary = {
        'bleu_scores': bleu_scores,
        'dataset_info': {
            'total_samples_evaluated': {
                strategy: len(evaluation_results[strategy]['predictions']) 
                for strategy in ['greedy', 'beam_search', 'top_k_sampling']
            },
            'model_path': MODEL_PATH,
            'config': config
        }
    }
    
    with open(bleu_summary_file, 'w') as f:
        json.dump(bleu_summary, f, indent=2)
    print(f"✅ BLEU scores summary saved to: {bleu_summary_file}")
    
    # Save comparison table as CSV if available
    if comparison_df is not None:
        csv_file = results_dir / "full_evaluation_comparison.csv"
        comparison_df.to_csv(csv_file, index=False, encoding='utf-8')
        print(f"✅ Full comparison table saved to: {csv_file}")
    
    # Save custom examples if any
    if custom_results:
        custom_file = results_dir / "custom_examples_results.json"
        with open(custom_file, 'w', encoding='utf-8') as f:
            json.dump(custom_results, f, ensure_ascii=False, indent=2)
        print(f"✅ Custom examples saved to: {custom_file}")
    
    # Create comprehensive summary report
    report_file = results_dir / "comprehensive_evaluation_report.txt"
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write("Transformer Model - Complete Dataset Evaluation Report\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Model: {MODEL_PATH}\n")
        f.write(f"Device: {device}\n")
        f.write(f"Source Language: {config['lang_src']}\n")
        f.write(f"Target Language: {config['lang_tgt']}\n")
        f.write(f"Sequence Length: {config['seq_len']}\n")
        f.write(f"Model Dimension: {config['d_model']}\n\n")
        
        if tokenizer_src and tokenizer_tgt:
            f.write("Vocabulary Sizes:\n")
            f.write(f"  Source: {tokenizer_src.get_vocab_size():,}\n")
            f.write(f"  Target: {tokenizer_tgt.get_vocab_size():,}\n\n")
        
        f.write("BLEU Scores (Complete Test Dataset):\n")
        f.write("-" * 40 + "\n")
        for strategy, score in bleu_scores.items():
            sample_count = len(evaluation_results[strategy]['predictions'])
            f.write(f"{strategy.upper():15}: {score:.4f} ({sample_count:,} samples)\n")
        
        f.write(f"\nBest performing strategy: {max(bleu_scores.items(), key=lambda x: x[1])[0].upper()}\n")
        f.write(f"Best BLEU score: {max(bleu_scores.values()):.4f}\n\n")
        
        f.write("Sample Translations (First 5):\n")
        f.write("-" * 30 + "\n")
        for i in range(min(5, len(evaluation_results['greedy']['predictions']))):
            f.write(f"\n{i+1}. Source: {evaluation_results['greedy']['sources'][i]}\n")
            f.write(f"   Reference: {' '.join(evaluation_results['greedy']['references'][i][0])}\n")
            f.write(f"   Greedy: {' '.join(evaluation_results['greedy']['predictions'][i])}\n")
            if i < len(evaluation_results['beam_search']['predictions']):
                f.write(f"   Beam: {' '.join(evaluation_results['beam_search']['predictions'][i])}\n")
            if i < len(evaluation_results['top_k_sampling']['predictions']):
                f.write(f"   Top-k: {' '.join(evaluation_results['top_k_sampling']['predictions'][i])}\n")
        
        if custom_results:
            f.write(f"\nCustom Examples ({len(custom_results)}):\n")
            f.write("-" * 20 + "\n")
            for i, result in enumerate(custom_results):
                f.write(f"\n{i+1}. Finnish: {result['finnish']}\n")
                f.write(f"   Greedy: {result['greedy']}\n")
                f.write(f"   Beam: {result['beam_search']}\n")
                f.write(f"   Top-k: {result['top_k']}\n")
    
    print(f"✅ Comprehensive report saved to: {report_file}")
    
    # Save configuration and model info
    info_file = results_dir / "model_and_config_info.json"
    info_data = {
        'model_path': MODEL_PATH,
        'config': config,
        'device': str(device),
        'vocab_sizes': {
            'source': tokenizer_src.get_vocab_size() if tokenizer_src else None,
            'target': tokenizer_tgt.get_vocab_size() if tokenizer_tgt else None
        },
        'evaluation_summary': {
            'strategies_tested': list(bleu_scores.keys()),
            'best_strategy': max(bleu_scores.items(), key=lambda x: x[1])[0] if bleu_scores else None,
            'best_bleu_score': max(bleu_scores.values()) if bleu_scores else 0.0
        }
    }
    
    with open(info_file, 'w') as f:
        json.dump(info_data, f, indent=2)
    print(f"✅ Model and config info saved to: {info_file}")
    
    print(f"\n📁 All results saved in: {results_dir.absolute()}")
    print("\nFiles created:")
    for file in sorted(results_dir.iterdir()):
        print(f"  - {file.name}")
    
    print(f"\n🎯 FINAL RESULTS SUMMARY:")
    print("=" * 50)
    for strategy, score in bleu_scores.items():
        sample_count = len(evaluation_results[strategy]['predictions'])
        print(f"{strategy.upper():15}: {score:.4f} ({sample_count:,} samples)")
    
    if bleu_scores:
        best_strategy = max(bleu_scores.items(), key=lambda x: x[1])
        print(f"\n🏆 Best Strategy: {best_strategy[0].upper()} (BLEU: {best_strategy[1]:.4f})")
        
else:
    print("❌ No evaluation results to save")

print("\n🎉 Complete dataset inference and evaluation finished!")

## Summary

This notebook demonstrates how to:

1. ✅ Load a pre-trained transformer model (.pt file)
2. ✅ Load complete test dataset from saved splits (as created by train.py)
3. ✅ Initialize tokenizers for Finnish and English
4. ✅ Implement three decoding strategies:
   - Greedy Decoding
   - Beam Search (beam_size=5)
   - Top-k Sampling (k=50, temperature=0.8)
5. ✅ **Run inference on the COMPLETE test dataset** (not just samples!)
6. ✅ Evaluate translation quality with BLEU scores for all strategies
7. ✅ Compare performance across all three decoding methods
8. ✅ Save comprehensive results for download

### Key Features:

- **Complete Dataset Evaluation**: Uses your actual dataset splits from training
- **Three Decoding Strategies**: Comprehensive comparison of different approaches
- **BLEU Score Analysis**: Corpus-level evaluation on the full test set
- **Progress Tracking**: Real-time progress bars and sample outputs
- **Comprehensive Reporting**: Detailed results with statistics and examples

### To use with your own data:

1. **Upload your files to Kaggle:**
   - Your trained model (.pt file)
   - Dataset splits folder (zip file) - **EXACTLY as created by train.py**
   - Source code files (utils.py, encoder.py, decoder.py, train.py, config.py)
   - Tokenizer files (tokenizer_fi.json, tokenizer_en.json)
   - Original dataset files (datasets/EUbookshop.fi, datasets/EUbookshop.en)

2. **Update file paths in Cell 3**

3. **Run all cells sequentially**

4. **Download results from the inference_results folder**

### Expected Output Files:
- `detailed_evaluation_results.json` - Complete predictions and references for all strategies
- `bleu_scores_summary.json` - BLEU scores and evaluation statistics
- `full_evaluation_comparison.csv` - Side-by-side comparison table
- `comprehensive_evaluation_report.txt` - Human-readable detailed report
- `custom_examples_results.json` - Additional custom translation examples
- `model_and_config_info.json` - Model configuration and metadata

### Performance Expectations:
- **Evaluation Time**: ~2-5 minutes per 1000 samples (depends on GPU)
- **Memory Usage**: Optimized for batch_size=1 inference
- **Output Quality**: Production-ready BLEU scores on complete test set

This notebook provides **research-grade evaluation** suitable for:
- Academic paper results
- Model comparison studies  
- Production deployment decisions
- Translation quality assessment

Happy evaluating! 🚀📊